<a href="https://colab.research.google.com/github/ghadirymaryam1-coder/PFMS-Net-Research2026/blob/main/PFMSPARALLEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, os, shutil

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Local free space (GB):", round(shutil.disk_usage("/content").free / 1e9, 2))
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

Python environment ready
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4
Local free space (GB): 75.31
Drive mounted: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

paths = {
    "GT CSV": "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv",
    "Vocabulary": "/content/drive/MyDrive/idpl_vocab.pt",
    "Images": "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

GT CSV: FOUND
Vocabulary: FOUND
Images: FOUND


In [7]:
%pip install --no-cache-dir --no-deps --force-reinstall "causal-conv1d==1.6.2.post1+cu.13.0.torch.2.11" --index-url https://wheels.astral.sh/simple/cu130/

Looking in indexes: https://wheels.astral.sh/simple/cu130/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 MB 155.5 MB/s  0:00:01


In [8]:
%pip install --no-cache-dir --no-deps --force-reinstall "mamba-ssm==2.3.2.post1+cu.13.0.torch.2.11" --index-url https://wheels.astral.sh/simple/cu130/

Looking in indexes: https://wheels.astral.sh/simple/cu130/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.4/349.4 MB 186.3 MB/s  0:00:01


In [10]:
%pip install --no-cache-dir --no-deps mambavision==1.2.0

In [11]:
from mambavision import create_model
import torch
import causal_conv1d
import mamba_ssm

model = create_model("mamba_vision_T", pretrained=False).cuda().eval()

with torch.no_grad():
    y = model(torch.randn(1, 3, 224, 224, device="cuda"))

print("MambaVision: OK")
print("GPU:", torch.cuda.get_device_name(0))
print("Output shape:", y.shape)
print("Finite output:", torch.isfinite(y).all().item())

del model, y
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


MambaVision: OK
GPU: Tesla T4
Output shape: torch.Size([1, 1000])
Finite output: True


In [12]:
import os

paths = {
    "GT": "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv",
    "Vocabulary": "/content/drive/MyDrive/idpl_vocab.pt",
    "Images": "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl",
}

for name, path in paths.items():
    print(f"{name}: {'OK' if os.path.exists(path) else 'NOT FOUND'}")

GT: OK
Vocabulary: OK
Images: OK


In [16]:
# ============================================================
# PFMS-Net Ablation | Cell 2: Spatial feature-map verification
# Real MambaVision-T + original IDPL-PFOD images
# ============================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from collections import Counter
from mambavision import create_model

GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
OUT_DIR = "/content/PFMS_ABLATION"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gt = pd.read_csv(GT_PATH)
train_df = gt[gt["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
test_df = gt[gt["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

# Inspect the saved vocabulary without assuming its internal format.
vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
print("Saved vocabulary object type:", type(vocab_obj).__name__)

if isinstance(vocab_obj, dict):
    print("Vocabulary keys (first 30):", list(vocab_obj.keys())[:30])
    for k, v in list(vocab_obj.items())[:10]:
        if isinstance(v, dict):
            print(f"  {k}: dict, entries={len(v)}")
        elif isinstance(v, (list, tuple)):
            print(f"  {k}: {type(v).__name__}, length={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__}", str(v)[:120])
elif isinstance(vocab_obj, (list, tuple)):
    print("Vocabulary length:", len(vocab_obj))
else:
    print("Vocabulary preview:", str(vocab_obj)[:500])

print(f"\nTrain rows: {len(train_df)} | Test rows: {len(test_df)}")
print("Unique training characters:", len(set("".join(train_df["text"].astype(str)))))

# Load a real image and inspect dimensions.
sample_path = str(train_df.iloc[0]["image_path"])
if not os.path.isfile(sample_path):
    raise FileNotFoundError(f"Image path from GT does not exist: {sample_path}")

with Image.open(sample_path) as im:
    im = im.convert("RGB")
    print("\nSample image:", os.path.basename(sample_path))
    print("Original image size (W,H):", im.size)

# Build the actual MambaVision-T backbone.
mamba = create_model("mamba_vision_T", pretrained=False).to(device).eval()
captured = {}

def capture_stage(name):
    def _hook(module, inputs, output):
        captured[name] = output
    return _hook

handles = []
for i, stage in enumerate(mamba.levels):
    handles.append(stage.register_forward_hook(capture_stage(f"stage_{i+1}")))

# Probe with the original aspect ratio scaled down only for this shape test.
# This is not training and does not resize any dataset files on disk.
probe_h, probe_w = 64, 448
x = torch.randn(1, 3, probe_h, probe_w, device=device)

with torch.no_grad():
    _ = mamba(x)

for h in handles:
    h.remove()

def describe_output(obj):
    if torch.is_tensor(obj):
        return f"Tensor{tuple(obj.shape)}"
    if isinstance(obj, (tuple, list)):
        return type(obj).__name__ + "[" + ", ".join(describe_output(z) for z in obj) + "]"
    return type(obj).__name__

print("\nMambaVision internal stage outputs:")
for name, obj in captured.items():
    print(name, "->", describe_output(obj))

# Choose a spatial tensor from the deepest stage output.
last_output = captured[f"stage_{len(mamba.levels)}"]
candidates = []
def collect_spatial(obj):
    if torch.is_tensor(obj) and obj.ndim == 4:
        candidates.append(obj)
    elif isinstance(obj, (tuple, list)):
        for z in obj:
            collect_spatial(z)
collect_spatial(last_output)

if not candidates:
    raise RuntimeError(
        "No 4D spatial feature map was exposed by the last stage. "
        "Do not train using the pooled 640-D classifier features."
    )

feature_map = max(candidates, key=lambda z: z.shape[1])
print("\nSelected spatial feature map:", tuple(feature_map.shape))
print("Spatial width (sequence axis):", feature_map.shape[-1])
print("Spatial height:", feature_map.shape[-2])
print("Feature channels:", feature_map.shape[1])

# Save only a small text report locally.
report_path = os.path.join(OUT_DIR, "mambavision_feature_probe.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n")
    f.write(f"Probe input: {tuple(x.shape)}\n")
    f.write(f"Selected feature map: {tuple(feature_map.shape)}\n")
    f.write(f"Vocabulary type: {type(vocab_obj).__name__}\n")

print("\nProbe report saved locally:", report_path)
print("No checkpoint or feature cache was written to Google Drive.")

Saved vocabulary object type: dict
Vocabulary keys (first 30): ['char2idx', 'idx2char', 'characters', 'PAD_IDX', 'BOS_IDX', 'EOS_IDX', 'UNK_IDX', 'vocab_size']
  char2idx: dict, entries=169
  idx2char: dict, entries=169
  characters: list, length=165
  PAD_IDX: int 0
  BOS_IDX: int 1
  EOS_IDX: int 2
  UNK_IDX: int 3
  vocab_size: int 169

Train rows: 23054 | Test rows: 4066
Unique training characters: 164

Sample image: 00490.tif
Original image size (W,H): (700, 50)

MambaVision internal stage outputs:
stage_1 -> Tensor(1, 160, 8, 56)
stage_2 -> Tensor(1, 320, 4, 28)
stage_3 -> Tensor(1, 640, 2, 14)
stage_4 -> Tensor(1, 640, 2, 14)

Selected spatial feature map: (1, 640, 2, 14)
Spatial width (sequence axis): 14
Spatial height: 2
Feature channels: 640

Probe report saved locally: /content/PFMS_ABLATION/mambavision_feature_probe.txt
No checkpoint or feature cache was written to Google Drive.


In [17]:
# ============================================================
# PFMS-Net Ablation | Cell 3: OCR data pipeline + CTC sanity test
# ============================================================
import os, gc, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from mambavision import create_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")
GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
OUT_DIR = "/content/PFMS_ABLATION"
os.makedirs(OUT_DIR, exist_ok=True)

gt = pd.read_csv(GT_PATH)
train_df = gt[gt["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
test_df = gt[gt["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

vocab = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
char2idx = vocab["char2idx"]
PAD_IDX = int(vocab["PAD_IDX"])
BOS_IDX = int(vocab["BOS_IDX"])
EOS_IDX = int(vocab["EOS_IDX"])
UNK_IDX = int(vocab["UNK_IDX"])
VOCAB_SIZE = int(vocab["vocab_size"])

# CTC blank must be distinct from actual target characters.
# PAD=0 is used as CTC blank; BOS/EOS are excluded from targets.
CTC_BLANK = PAD_IDX

# Resize the line to a fixed landscape size, keeping the line layout.
# Image files on Drive are never modified.
IMG_H, IMG_W = 64, 896
transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

class IDPLLineDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = str(row["image_path"])
        text = str(row["text"]).replace("\n", " ").strip()
        with Image.open(path) as im:
            image = transform(im.convert("RGB"))

        # CTC targets: character IDs only, no BOS/EOS.
        target = [char2idx.get(ch, UNK_IDX) for ch in text]
        target = [t for t in target if t not in (PAD_IDX, BOS_IDX, EOS_IDX)]
        return image, torch.tensor(target, dtype=torch.long), text, path

def ctc_collate(batch):
    images, targets, texts, paths = zip(*batch)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets) if target_lengths.sum().item() else torch.empty(0, dtype=torch.long)
    return torch.stack(images), flat_targets, target_lengths, list(texts), list(paths)

# Keep test data separate; no test examples used for optimization.
train_ds = IDPLLineDataset(train_df)
test_ds = IDPLLineDataset(test_df)

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=2,
    collate_fn=ctc_collate, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=4, shuffle=False, num_workers=2,
    collate_fn=ctc_collate, pin_memory=True
)

# Real MambaVision backbone
backbone = create_model("mamba_vision_T", pretrained=False).to(device).eval()
stage4 = backbone.levels[-1]
captured = {}

def save_stage_output(module, inputs, output):
    captured["feat"] = output

hook = stage4.register_forward_hook(save_stage_output)

# Use one real batch to verify CTC dimensions before training.
images, targets, target_lengths, texts, paths = next(iter(train_loader))
images = images.to(device, non_blocking=True)

with torch.no_grad():
    _ = backbone(images)
hook.remove()

feat = captured["feat"]
if not torch.is_tensor(feat) or feat.ndim != 4:
    raise RuntimeError(f"Expected spatial feature map [B,C,H,W], got {type(feat)}")

# Convert spatial map [B,C,H,W] -> sequence [T,B,C].
# Pool only vertical dimension, preserving the horizontal sequence.
feat = feat.mean(dim=2).permute(2, 0, 1).contiguous()
sequence_length = feat.shape[0]
batch_size = feat.shape[1]

print("Vocabulary size:", VOCAB_SIZE)
print("Image batch:", tuple(images.shape))
print("MambaVision sequence [T,B,C]:", tuple(feat.shape))
print("CTC blank index:", CTC_BLANK)
print("Target lengths:", target_lengths.tolist())
print("Text samples:", texts[:2])

if sequence_length < int(target_lengths.max().item()):
    print(
        "\nWARNING: CTC sequence is shorter than at least one target. "
        "Do not start training with this output stride. We must increase "
        "the horizontal sequence resolution before training."
    )
else:
    logits = torch.randn(sequence_length, batch_size, VOCAB_SIZE, device=device)
    log_probs = logits.log_softmax(dim=-1)
    input_lengths = torch.full((batch_size,), sequence_length, dtype=torch.long)
    loss_fn = nn.CTCLoss(blank=CTC_BLANK, zero_infinity=True)
    loss = loss_fn(log_probs, targets.to(device), input_lengths, target_lengths)
    print("CTC sanity loss finite:", bool(torch.isfinite(loss)))
    print("CTC sanity loss:", float(loss))

del backbone, images, feat, captured
gc.collect()
torch.cuda.empty_cache()
print("\nData pipeline verified. No training was started in this cell.")

Vocabulary size: 169
Image batch: (4, 3, 64, 896)
MambaVision sequence [T,B,C]: (28, 4, 640)
CTC blank index: 0
Target lengths: [67, 77, 72, 72]
Text samples: ['دشمنان در كمین هستند و تا ظهور امام عصر این انقلاب را به پیش ببریم.', 'به شهروندان مهم و قابل تحسین دانست. به گزارش تحریرنو از روابط عمومی بانک شهر،']


Data pipeline verified. No training was started in this cell.


In [18]:
# ============================================================
# MambaVision OCR: find a suitable spatial stage for CTC
# No training and no Drive writes
# ============================================================
import torch
import gc
from mambavision import create_model

device = "cuda"
model = create_model("mamba_vision_T", pretrained=False).to(device).eval()

captured = {}
handles = []

def make_hook(name):
    def hook(module, inputs, output):
        captured[name] = output
    return hook

for i, stage in enumerate(model.levels):
    handles.append(stage.register_forward_hook(make_hook(f"stage_{i+1}")))

# 64x896 line image, matching the tested OCR input shape
x = torch.randn(1, 3, 64, 896, device=device)

with torch.no_grad():
    _ = model(x)

for h in handles:
    h.remove()

print("Stage | Feature map [B,C,H,W] | CTC steps after vertical pooling")
for name, value in captured.items():
    if torch.is_tensor(value) and value.ndim == 4:
        print(f"{name}: {tuple(value.shape)} | T={value.shape[-1]}")

# Stage 2 is the first candidate: higher horizontal resolution
feat = captured["stage_2"]
assert feat.ndim == 4, f"Unexpected stage_2 output: {type(feat)}"
print("\nCandidate stage_2 channels:", feat.shape[1])
print("Candidate CTC steps:", feat.shape[-1])
print("Longest sample target length in previous batch: 77")
print("CTC minimum feasibility requires input steps >= target length.")
print("For repeated adjacent characters, additional steps may be required.")

del model, x, feat, captured
gc.collect()
torch.cuda.empty_cache()

Stage | Feature map [B,C,H,W] | CTC steps after vertical pooling
stage_1: (1, 160, 8, 112) | T=112
stage_2: (1, 320, 4, 56) | T=56
stage_3: (1, 640, 2, 28) | T=28
stage_4: (1, 640, 2, 28) | T=28

Candidate stage_2 channels: 320
Candidate CTC steps: 56
Longest sample target length in previous batch: 77
CTC minimum feasibility requires input steps >= target length.
For repeated adjacent characters, additional steps may be required.


In [19]:
# ============================================================
# PFMS-Net | MambaVision spatial-resolution check
# ============================================================
import torch
import pandas as pd
from mambavision import create_model

GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
df = pd.read_csv(GT_PATH)
train_df = df[df["split"].astype(str).str.lower() == "train"].copy()

vocab = torch.load(
    "/content/drive/MyDrive/idpl_vocab.pt",
    map_location="cpu",
    weights_only=False
)
char2idx = vocab["char2idx"]
excluded = {
    int(vocab["PAD_IDX"]),
    int(vocab["BOS_IDX"]),
    int(vocab["EOS_IDX"])
}
unk = int(vocab["UNK_IDX"])

def target_len(text):
    return sum(
        1 for ch in str(text).replace("\n", " ").strip()
        if char2idx.get(ch, unk) not in excluded
    )

lengths = train_df["text"].map(target_len)
print("Train samples:", len(train_df))
print("Target length max:", int(lengths.max()))
print("Target length p99:", int(lengths.quantile(0.99)))
print("Targets longer than 128:", int((lengths > 128).sum()))

model = create_model("mamba_vision_T", pretrained=False).cuda().eval()
captured = {}

def hook_stage1(module, inputs, output):
    captured["stage1"] = output

h = model.levels[0].register_forward_hook(hook_stage1)
x = torch.randn(1, 3, 64, 1024, device="cuda")

with torch.no_grad():
    _ = model(x)

h.remove()
feat = captured["stage1"]
print("Stage-1 spatial feature map:", tuple(feat.shape))
print("CTC sequence length:", int(feat.shape[-1]))
print("CTC blank:", vocab["PAD_IDX"])

assert feat.ndim == 4
assert feat.shape[-1] >= int(lengths.max()), (
    "CTC sequence is shorter than the longest target. "
    "Increase width or use a different spatial feature strategy."
)

print("Resolution check passed for target lengths.")
del model, x, feat, captured
torch.cuda.empty_cache()

Train samples: 23054
Target length max: 114
Target length p99: 97
Targets longer than 128: 0
Stage-1 spatial feature map: (1, 160, 8, 128)
CTC sequence length: 128
CTC blank: 0
Resolution check passed for target lengths.


In [23]:

# CELL 1/2 — Complete PFMS-Net ablation setup and preflight
import os, sys, gc, json, time, random, subprocess
from pathlib import Path

def install_if_missing(module, package):
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_if_missing("timm", "timm")
install_if_missing("mambavision", "mambavision")

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from mambavision import create_model

# ---------------- CONFIG ----------------
SEED = 42
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-2
IMG_H, IMG_W = 64, 1024
SEQ_LEN = 128
D_MODEL = 256
NUM_WORKERS = 2
USE_AMP = True

CSV_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
OUT_DIR = Path("/content/PFMS_ABLATION_FINAL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU فعال نیست؛ در Colab GPU را فعال کنید."
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

# ---------------- VOCABULARY ----------------
vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
assert isinstance(vocab_obj, dict) and "char2idx" in vocab_obj, (
    "ساختار فایل واژگان با کلید char2idx مطابقت ندارد."
)

char2idx = {str(k): int(v) for k, v in vocab_obj["char2idx"].items()}
raw_idx2char = vocab_obj.get("idx2char")
idx2char = (
    {int(k): str(v) for k, v in raw_idx2char.items()}
    if raw_idx2char is not None
    else {v: k for k, v in char2idx.items()}
)
PAD_IDX = int(vocab_obj.get("PAD_IDX", 0))
BOS_IDX = int(vocab_obj.get("BOS_IDX", 1))
EOS_IDX = int(vocab_obj.get("EOS_IDX", 2))
UNK_IDX = int(vocab_obj.get("UNK_IDX", 3))
VOCAB_SIZE = max(max(char2idx.values()), max(idx2char.keys())) + 1
assert (PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX) == (0, 1, 2, 3)
assert VOCAB_SIZE == 169, f"Expected 169 tokens, found {VOCAB_SIZE}"
BLANK_IDX = PAD_IDX
SPECIAL_IDS = {PAD_IDX, BOS_IDX, EOS_IDX}

def encode_text(text):
    ids = [char2idx.get(ch, UNK_IDX) for ch in str(text)]
    ids = [i for i in ids if i not in SPECIAL_IDS]
    if not ids:
        raise ValueError(f"Empty target after special-token removal: {text!r}")
    if any(i < 0 or i >= VOCAB_SIZE for i in ids):
        raise ValueError("Target token outside vocabulary.")
    return ids

def ctc_min_steps(ids):
    return len(ids) + sum(ids[i] == ids[i-1] for i in range(1, len(ids)))

def decode_ctc(ids):
    # Greedy CTC decoding: collapse repeated tokens, then remove blanks.
    result, prev = [], None
    for token in ids:
        token = int(token)
        if token != prev and token != BLANK_IDX:
            result.append(idx2char.get(token, "�"))
        prev = token
    return "".join(result)

# ---------------- DATA SPLIT ----------------
df = pd.read_csv(CSV_PATH)
required = {"split", "image_path", "image_name", "text"}
assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
df["split"] = df["split"].astype(str).str.lower().str.strip()
df["text"] = df["text"].fillna("").astype(str)

def resolve_path(row):
    candidates = [
        Path(str(row["image_path"])),
        Path(IMAGE_ROOT) / str(row["image_name"]),
        Path(IMAGE_ROOT) / Path(str(row["image_path"])).name,
    ]
    for p in candidates:
        if p.is_file():
            return str(p)
    return str(candidates[0])

df["resolved_path"] = df.apply(resolve_path, axis=1)
df = df[df["text"].str.len() > 0].copy()
missing = df[~df["resolved_path"].map(lambda p: Path(p).is_file())]
if len(missing):
    raise FileNotFoundError(
        f"{len(missing)} image files missing. Examples:\n" +
        "\n".join(missing["resolved_path"].head(5))
    )

train_all = df[df["split"].str.contains("train")].copy()
test_df = df[df["split"].str.contains("test")].copy()
assert len(train_all) > 0 and len(test_df) > 0

# Deterministic 90/10 train/validation split. Test remains untouched.
perm = np.random.RandomState(SEED).permutation(len(train_all))
n_val = max(1, int(round(0.10 * len(train_all))))
val_df = train_all.iloc[perm[:n_val]].reset_index(drop=True)
train_df = train_all.iloc[perm[n_val:]].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for split_name, part in [
    ("train", train_df), ("validation", val_df), ("test", test_df)
]:
    targets = [encode_text(t) for t in part["text"]]
    max_target = max(map(len, targets))
    max_required = max(map(ctc_min_steps, targets))
    print(f"{split_name}: n={len(part)}, max_target={max_target}, "
          f"max_CTC_min_steps={max_required}")
    if max_required > SEQ_LEN:
        raise ValueError(
            f"{split_name}: CTC needs {max_required} steps; "
            f"configured sequence length is {SEQ_LEN}."
        )

print("Vocabulary:", VOCAB_SIZE, "| CTC blank:", BLANK_IDX)
print("Train/validation/test:", len(train_df), len(val_df), len(test_df))

# ---------------- IMAGE DATASET ----------------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class PersianLineDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row["resolved_path"]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im = im.resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
            arr = np.asarray(im, dtype=np.float32) / 255.0
        image = torch.from_numpy(arr).permute(2, 0, 1)
        image = (image - MEAN) / STD
        target = torch.tensor(encode_text(row["text"]), dtype=torch.long)
        return image, target, str(row["text"]), str(row["image_name"])

def collate_lines(batch):
    images, targets, texts, names = zip(*batch)
    lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    return torch.stack(images), torch.cat(targets), lengths, list(texts), list(names)

def make_loader(frame, shuffle):
    return DataLoader(
        PersianLineDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        collate_fn=collate_lines,
    )

train_loader = make_loader(train_df, True)
val_loader = make_loader(val_df, False)
test_loader = make_loader(test_df, False)

# ---------------- REAL SWIN ENCODER ----------------
class SwinStage1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(0,),
            img_size=(IMG_H, IMG_W),
        )
        self.channels = int(self.net.feature_info.channels()[0])

    def forward(self, x):
        feat = self.net(x)[0]
        if feat.ndim != 4:
            raise RuntimeError(f"Swin expected 4D features; got {tuple(feat.shape)}")

        # Correctly handle NHWC and NCHW layouts.
        if feat.shape[-1] == self.channels:
            feat = feat.permute(0, 3, 1, 2).contiguous()
        elif feat.shape[1] == self.channels:
            pass
        else:
            raise RuntimeError(
                f"Swin channel axis unclear: shape={tuple(feat.shape)}, "
                f"channels={self.channels}"
            )
        return feat

# ---------------- REAL MAMBAVISION ENCODER ----------------
class RealMambaVisionStage1(nn.Module):
    """
    Actual MambaVision-T stem + stage 1, not its 1000-class classifier.
    This is a stage-1 feature encoder, not the complete four-stage backbone.
    """
    def __init__(self):
        super().__init__()
        backbone = create_model("mamba_vision_T", pretrained=False)
        if not hasattr(backbone, "patch_embed") or not hasattr(backbone, "levels"):
            raise RuntimeError(
                "Installed MambaVision API changed: patch_embed/levels unavailable."
            )
        self.patch_embed = backbone.patch_embed
        self.stage1 = backbone.levels[0]
        self.channels = 160
        del backbone

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.stage1(x)
        if x.ndim != 4:
            raise RuntimeError(f"MambaVision stage-1 output is not 4D: {tuple(x.shape)}")
        if x.shape[1] != self.channels:
            raise RuntimeError(
                f"Expected MambaVision NCHW with {self.channels} channels; "
                f"got {tuple(x.shape)}"
            )
        return x

# ---------------- FUSION AND CTC CLASSIFIER ----------------
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=D_MODEL, heads=4):
        super().__init__()
        self.qnorm = nn.LayerNorm(dim)
        self.kvnorm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=0.1)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, 2 * dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2 * dim, dim),
        )
        self.drop = nn.Dropout(0.1)

    def forward(self, q, kv):
        z, _ = self.attn(
            self.qnorm(q), self.kvnorm(kv), self.kvnorm(kv),
            need_weights=False
        )
        q = q + self.drop(z)
        return q + self.drop(self.ffn(q))

class OCRAblationModel(nn.Module):
    VALID = {"swin_only", "mamba_only", "simple_fusion",
             "cross_attention", "full_pfms"}

    def __init__(self, variant):
        super().__init__()
        if variant not in self.VALID:
            raise ValueError(f"Unknown variant: {variant}")
        self.variant = variant

        self.swin = SwinStage1() if variant != "mamba_only" else None
        self.mamba = RealMambaVisionStage1() if variant != "swin_only" else None

        if self.swin is not None:
            self.swin_proj = nn.Conv2d(self.swin.channels, D_MODEL, kernel_size=1)
        if self.mamba is not None:
            self.mamba_proj = nn.Conv2d(self.mamba.channels, D_MODEL, kernel_size=1)

        if variant == "simple_fusion":
            self.fuse = nn.Sequential(
                nn.Linear(2 * D_MODEL, D_MODEL),
                nn.GELU(),
                nn.LayerNorm(D_MODEL),
            )
        if variant in {"cross_attention", "full_pfms"}:
            self.swin_to_mamba = CrossAttentionBlock()
            self.mamba_to_swin = CrossAttentionBlock()

        if variant == "full_pfms":
            self.refine = nn.Sequential(
                nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
                nn.GELU(),
                nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
            )
        else:
            self.refine = None

        self.classifier = nn.Linear(D_MODEL, VOCAB_SIZE)

    @staticmethod
    def spatial_to_sequence(feat, projection):
        feat = projection(feat)       # [B,D,H,W]
        feat = feat.mean(dim=2)       # [B,D,W]
        feat = F.adaptive_avg_pool1d(feat, SEQ_LEN)
        return feat.transpose(1, 2).contiguous()  # [B,T,D]

    def forward(self, images):
        s = None
        m = None
        if self.swin is not None:
            s = self.spatial_to_sequence(self.swin(images), self.swin_proj)
        if self.mamba is not None:
            m = self.spatial_to_sequence(self.mamba(images), self.mamba_proj)

        if self.variant == "swin_only":
            z = s
        elif self.variant == "mamba_only":
            z = m
        elif self.variant == "simple_fusion":
            z = self.fuse(torch.cat((s, m), dim=-1))
        else:
            s2 = self.swin_to_mamba(s, m)
            m2 = self.mamba_to_swin(m, s)
            z = 0.5 * (s2 + m2)
            if self.refine is not None:
                z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)

        logits = self.classifier(z)
        if logits.shape[1:] != (SEQ_LEN, VOCAB_SIZE):
            raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")
        return logits

# ---------------- PREFLIGHT: ALL FIVE MODELS ----------------
def run_preflight():
    results = []
    variants = ["swin_only", "mamba_only", "simple_fusion",
                "cross_attention", "full_pfms"]
    x = torch.zeros(1, 3, IMG_H, IMG_W, device=DEVICE)

    for variant in variants:
        gc.collect()
        torch.cuda.empty_cache()
        model = OCRAblationModel(variant).to(DEVICE).eval()
        try:
            with torch.inference_mode():
                y = model(x)
            assert tuple(y.shape) == (1, SEQ_LEN, VOCAB_SIZE), (
                variant, tuple(y.shape)
            )
            n_total = sum(p.numel() for p in model.parameters())
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            row = {
                "variant": variant,
                "logits_shape": str(tuple(y.shape)),
                "parameters_total": n_total,
                "parameters_trainable": n_train,
            }
            results.append(row)
            print(
                f"PREFLIGHT OK: {variant} | shape={tuple(y.shape)} | "
                f"trainable={n_train:,}"
            )
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

    pd.DataFrame(results).to_csv(OUT_DIR / "preflight.csv", index=False)
    return results

PREFLIGHT_RESULTS = run_preflight()
print("Preflight passed. No training has been run in Cell 1.")


PyTorch: 2.11.0+cu130 | GPU: Tesla T4
train: n=20749, max_target=115, max_CTC_min_steps=116
validation: n=2305, max_target=109, max_CTC_min_steps=111
test: n=4066, max_target=105, max_CTC_min_steps=107
Vocabulary: 169 | CTC blank: 0
Train/validation/test: 20749 2305 4066
PREFLIGHT OK: swin_only | shape=(1, 128, 169) | trainable=297,855
PREFLIGHT OK: mamba_only | shape=(1, 128, 169) | trainable=339,657
PREFLIGHT OK: simple_fusion | shape=(1, 128, 169) | trainable=725,919
PREFLIGHT OK: cross_attention | shape=(1, 128, 169) | trainable=1,649,311
PREFLIGHT OK: full_pfms | shape=(1, 128, 169) | trainable=2,043,039
Preflight passed. No training has been run in Cell 1.


In [ ]:

# CELL 2/2 — Training, held-out evaluation, checkpoints and CSV tables
import gc, json, time
from pathlib import Path

# Stop immediately if Cell 1 did not finish successfully.
assert "PREFLIGHT_RESULTS" in globals(), (
    "ابتدا سلول اول را کامل اجرا کنید؛ PREFLIGHT_RESULTS وجود ندارد."
)
assert len(PREFLIGHT_RESULTS) == 5, "پیش‌آزمون هر پنج مدل موفق نشده است."
assert all(r["logits_shape"] == str((1, SEQ_LEN, VOCAB_SIZE))
           for r in PREFLIGHT_RESULTS)

VARIANTS = [
    "swin_only",
    "mamba_only",
    "simple_fusion",
    "cross_attention",
    "full_pfms",
]

ctc_loss_fn = nn.CTCLoss(
    blank=BLANK_IDX,
    reduction="mean",
    zero_infinity=False,
)

def edit_distance(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j - 1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]

def calculate_metrics(refs, preds):
    char_edits = sum(edit_distance(list(r), list(p)) for r, p in zip(refs, preds))
    ref_chars = sum(max(1, len(r)) for r in refs)
    word_edits = sum(
        edit_distance(r.split(), p.split()) for r, p in zip(refs, preds)
    )
    ref_words = sum(max(1, len(r.split())) for r in refs)
    exact = sum(r == p for r, p in zip(refs, preds))
    cer = char_edits / max(1, ref_chars)
    wer = word_edits / max(1, ref_words)
    return {
        "CER": cer,
        "WER": wer,
        "character_accuracy_1_minus_CER": 1.0 - cer,
        "word_accuracy_1_minus_WER": 1.0 - wer,
        "sequence_accuracy": exact / max(1, len(refs)),
        "char_edits": char_edits,
        "reference_characters": ref_chars,
        "word_edits": word_edits,
        "reference_words": ref_words,
        "exact_sequences": exact,
        "samples": len(refs),
    }

def amp_context():
    return torch.autocast(
        device_type="cuda", dtype=torch.float16, enabled=USE_AMP
    )

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def evaluate_model(model, loader, save_predictions=False):
    model.eval()
    loss_sum, n_seen, elapsed = 0.0, 0, 0.0
    refs, preds, rows = [], [], []

    with torch.inference_mode():
        for images, targets, target_lengths, texts, names in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)
            bsz = images.size(0)

            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with amp_context():
                logits = model(images)
            torch.cuda.synchronize()
            elapsed += time.perf_counter() - t0

            log_probs = logits.float().log_softmax(dim=-1)
            input_lengths = torch.full(
                (bsz,), log_probs.size(1), dtype=torch.long
            )
            loss = ctc_loss_fn(
                log_probs.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths.cpu(),
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite evaluation CTC loss.")

            loss_sum += loss.item() * bsz
            n_seen += bsz
            batch_preds = [
                decode_ctc(seq) for seq in log_probs.argmax(-1).cpu().tolist()
            ]
            refs.extend(texts)
            preds.extend(batch_preds)

            if save_predictions:
                rows.extend({
                    "image_name": name,
                    "reference": ref,
                    "prediction": pred,
                    "exact_match": (ref == pred),
                } for name, ref, pred in zip(names, texts, batch_preds))

    metrics = calculate_metrics(refs, preds)
    metrics["loss"] = loss_sum / max(1, n_seen)
    metrics["inference_ms_per_image"] = 1000 * elapsed / max(1, n_seen)
    metrics["images_per_second"] = n_seen / max(elapsed, 1e-12)

    if save_predictions:
        return metrics, pd.DataFrame(rows)
    return metrics

def train_variant(variant):
    print("\n" + "=" * 70)
    print("START:", variant)
    print("=" * 70)

    seed_everything(SEED)
    model = OCRAblationModel(variant).to(DEVICE)
    params_total = count_params(model)
    params_trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    model_dir = OUT_DIR / variant
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "best_model.pt"
    last_path = model_dir / "last_model.pt"
    history = []
    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, n_seen = 0.0, 0
        start_epoch = time.perf_counter()

        for step, (images, targets, target_lengths, texts, names) in enumerate(
            train_loader, start=1
        ):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)
            bsz = images.size(0)

            # Check CTC alignment requirement for every sample in this batch.
            needed = max(ctc_min_steps(encode_text(t)) for t in texts)
            if needed > SEQ_LEN:
                raise RuntimeError(
                    f"CTC alignment invalid: batch needs {needed} steps, "
                    f"but output has {SEQ_LEN}."
                )

            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = model(images)

            log_probs = logits.float().log_softmax(dim=-1)
            input_lengths = torch.full(
                (bsz,), log_probs.size(1), dtype=torch.long
            )
            loss = ctc_loss_fn(
                log_probs.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths.cpu(),
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite training loss at {variant}, "
                    f"epoch {epoch}, batch {step}."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item() * bsz
            n_seen += bsz

            if step % 100 == 0 or step == len(train_loader):
                print(
                    f"{variant} | epoch {epoch}/{EPOCHS} | "
                    f"batch {step}/{len(train_loader)} | loss={loss.item():.5f}"
                )

        train_loss = loss_sum / max(1, n_seen)
        val_metrics = evaluate_model(model, val_loader)
        epoch_seconds = time.perf_counter() - start_epoch

        record = {
            "variant": variant,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_CER": val_metrics["CER"],
            "val_WER": val_metrics["WER"],
            "val_sequence_accuracy": val_metrics["sequence_accuracy"],
            "epoch_seconds": epoch_seconds,
            "parameters_total": params_total,
            "parameters_trainable": params_trainable,
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(
            model_dir / "training_history.csv", index=False
        )

        state = {
            "variant": variant,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_metrics["loss"],
            "seed": SEED,
            "vocab_size": VOCAB_SIZE,
            "blank_idx": BLANK_IDX,
            "sequence_length": SEQ_LEN,
            "image_height": IMG_H,
            "image_width": IMG_W,
        }
        torch.save(state, last_path)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(state, best_path)

        print(
            f"Epoch {epoch}: train_loss={train_loss:.5f} | "
            f"val_loss={val_metrics['loss']:.5f} | "
            f"val_CER={val_metrics['CER']:.5f} | "
            f"val_WER={val_metrics['WER']:.5f} | "
            f"{epoch_seconds:.1f}s"
        )

    del optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    # Final held-out evaluation uses the best validation checkpoint.
    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state_dict"])
    test_metrics, pred_df = evaluate_model(
        model, test_loader, save_predictions=True
    )

    pred_file = model_dir / "test_predictions.csv"
    pred_df.to_csv(pred_file, index=False)

    summary = {
        "variant": variant,
        "best_epoch": int(best["epoch"]),
        "best_val_loss": float(best["val_loss"]),
        "parameters_total": params_total,
        "parameters_trainable": params_trainable,
        **{f"test_{k}": v for k, v in test_metrics.items()},
        "seed": SEED,
        "epochs_requested": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "image_height": IMG_H,
        "image_width": IMG_W,
        "ctc_sequence_length": SEQ_LEN,
        "vocab_size": VOCAB_SIZE,
        "checkpoint": str(best_path),
        "prediction_file": str(pred_file),
    }

    with open(model_dir / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("TEST RESULTS:", json.dumps(summary, ensure_ascii=False, indent=2))
    del model, best
    gc.collect()
    torch.cuda.empty_cache()
    return summary

# Record protocol before the long-running experiment.
config = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "loss": "CTCLoss",
    "ctc_blank": BLANK_IDX,
    "image_size": [IMG_H, IMG_W],
    "sequence_length": SEQ_LEN,
    "vocab_size": VOCAB_SIZE,
    "variants": VARIANTS,
    "mamba_encoder": "MambaVision-T stem + actual stage 1",
    "swin_encoder": "timm Swin-T stage-1 spatial features",
    "validation": "deterministic 10% split from official training data",
    "test": "held out from training and model selection",
}
with open(OUT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

summaries = []
for variant in VARIANTS:
    result = train_variant(variant)
    summaries.append(result)
    pd.DataFrame(summaries).to_csv(
        OUT_DIR / "ablation_results_partial.csv", index=False
    )

results = pd.DataFrame(summaries)
results.to_csv(OUT_DIR / "ablation_results.csv", index=False)

paper_table = results[[
    "variant", "parameters_total",
    "test_CER", "test_WER",
    "test_character_accuracy_1_minus_CER",
    "test_word_accuracy_1_minus_WER",
    "test_sequence_accuracy",
    "test_inference_ms_per_image",
    "test_images_per_second",
]].copy()

percentage_cols = [
    "test_CER", "test_WER",
    "test_character_accuracy_1_minus_CER",
    "test_word_accuracy_1_minus_WER",
    "test_sequence_accuracy",
]
for col in percentage_cols:
    paper_table[col] *= 100.0

paper_table = paper_table.rename(columns={
    "variant": "Model",
    "parameters_total": "Parameters",
    "test_CER": "CER (%)",
    "test_WER": "WER (%)",
    "test_character_accuracy_1_minus_CER": "1-CER (%)",
    "test_word_accuracy_1_minus_WER": "1-WER (%)",
    "test_sequence_accuracy": "Sequence accuracy (%)",
    "test_inference_ms_per_image": "Inference ms/image",
    "test_images_per_second": "Images/s",
})
paper_table.to_csv(OUT_DIR / "ablation_paper_table.csv", index=False)

print("\nFINAL MEASURED RESULTS")
display(paper_table.round(4))
print("\nOutput directory:", OUT_DIR)



START: swin_only
swin_only | epoch 1/10 | batch 100/5188 | loss=3.19267
swin_only | epoch 1/10 | batch 200/5188 | loss=3.07598
swin_only | epoch 1/10 | batch 300/5188 | loss=3.14058
swin_only | epoch 1/10 | batch 400/5188 | loss=3.05177
swin_only | epoch 1/10 | batch 500/5188 | loss=3.00459
swin_only | epoch 1/10 | batch 600/5188 | loss=3.07159
swin_only | epoch 1/10 | batch 700/5188 | loss=3.17005
swin_only | epoch 1/10 | batch 800/5188 | loss=3.01859
swin_only | epoch 1/10 | batch 900/5188 | loss=3.20653
swin_only | epoch 1/10 | batch 1000/5188 | loss=2.97746
swin_only | epoch 1/10 | batch 1100/5188 | loss=3.23462
swin_only | epoch 1/10 | batch 1200/5188 | loss=3.02298
swin_only | epoch 1/10 | batch 1300/5188 | loss=3.06057
swin_only | epoch 1/10 | batch 1400/5188 | loss=3.04255
swin_only | epoch 1/10 | batch 1500/5188 | loss=2.96517
swin_only | epoch 1/10 | batch 1600/5188 | loss=2.99148
swin_only | epoch 1/10 | batch 1700/5188 | loss=3.07707
swin_only | epoch 1/10 | batch 1800/518